# NI v3 builder smoke test

Exercises the rewritten `NormalisationImputationDataset` against the new converter-canonical wire schema. Most cells are **offline** — they construct a builder and print `job_run_params` for inspection. The last section is optional and submits a few jobs to a live API.

Coverage:
- protein × {median + centre_at_zero, quantile + include_imputed_values, sum, batch correction (combat / limma)}
- peptide × {by missing values, by ptm localization probability}
- gene × {cpm + prior_count, batch correction (combat seq), by minimum abundance}
- imputation × {mnar, knn, knn_tn, mindet, set to constant, set to missing, skip}
- `filter_only(...)` classmethod
- legacy underscored aliases (backward compat)
- validation rejections

In [ ]:
import json
from uuid import UUID

from md_python.models.dataset_builders import NormalisationImputationDataset

DUMMY_INPUT = [str(UUID(int=1))]
DESIGN = {
    "sample_name": ["s1", "s2", "s3", "s4"],
    "condition": ["a", "a", "b", "b"],
    "batch": ["x", "y", "x", "y"],
}


def show(label, ni):
    """Validate, build, and pretty-print job_run_params for visual inspection."""
    ni.validate()
    p = ni.to_dataset().job_run_params
    print(f"=== {label} ===")
    print(json.dumps(p, indent=2, default=str))
    print()
    return p


print(NormalisationImputationDataset.help())

## 1. Protein × normalisation methods

In [ ]:
p = show(
    "protein + median (defaults)",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="protein median",
        normalisation_method="median",
        imputation_method="skip",
    ),
)
assert p["median_normalisation_centre_at_zero"] is True
assert p["include_imputed_values"] is False
assert p["normalisation_methods_proteomics"] == "median"
assert p["filtration_methods_protein"] == "skip"
assert p["imputation_methods"] == "skip"

p = show(
    "protein + median (centre_at_zero=False)",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="protein median noctr",
        normalisation_method="median",
        imputation_method="skip",
        median_normalisation_centre_at_zero=False,
    ),
)
assert p["median_normalisation_centre_at_zero"] is False

p = show(
    "protein + quantile + include_imputed_values=True",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="protein quantile",
        normalisation_method="quantile",
        imputation_method="skip",
        include_imputed_values=True,
    ),
)
assert p["include_imputed_values"] is True

p = show(
    "protein + sum",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="protein sum",
        normalisation_method="sum",
        imputation_method="skip",
    ),
)
assert p["normalisation_methods_proteomics"] == "sum"
assert p["include_imputed_values"] is False

## 2. Protein × batch correction (combat / limma)

In [ ]:
p = show(
    "protein + ComBat (mean_only=True, ref_batch=x)",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="protein combat",
        normalisation_method="batch correction",
        imputation_method="skip",
        batch_correction_technique="combat",
        batch_variable_combat="batch",
        mean_only=True,
        reference_batch_combat="x",
        design_variables=[{"column": "condition", "type": "categorical"}],
        experiment_design=DESIGN,
    ),
)
assert p["batch_correction_technique_proteomics"] == "combat"
assert p["batch_variable_combat"] == "batch"
assert p["mean_only"] is True
assert p["reference_batch_combat"] == "x"
assert "batch_variables" not in p

p = show(
    "protein + Limma remove batch effect",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="protein limma",
        normalisation_method="batch correction",
        imputation_method="skip",
        batch_correction_technique="limma remove batch effect",
        batch_variables=[{"column": "batch", "type": "categorical"}],
        design_variables=[{"column": "condition", "type": "categorical"}],
        experiment_design=DESIGN,
    ),
)
assert p["batch_correction_technique_proteomics"] == "limma remove batch effect"
assert p["batch_variables"] == [{"column": "batch", "type": "categorical"}]
assert "batch_variable_combat" not in p
assert "mean_only" not in p

## 3. Imputation methods (including knn_tn and mindet)

In [ ]:
p = show(
    "imputation: mnar (defaults)",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="mnar",
        normalisation_method="skip",
        imputation_method="mnar",
    ),
)
assert p["std_position"] == 1.8
assert p["std_width"] == 0.3

p = show(
    "imputation: knn (defaults)",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="knn",
        normalisation_method="skip",
        imputation_method="knn",
    ),
)
assert p["n_neighbors"] == 3
assert p["weights"] == "uniform"

p = show(
    "imputation: knn_tn (defaults)",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="knn_tn",
        normalisation_method="skip",
        imputation_method="knn_tn",
    ),
)
assert p["knn_tn_k"] == 5
assert p["knn_tn_distance"] == "truncation"

p = show(
    "imputation: knn_tn (custom k=4 distance=correlation)",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="knn_tn custom",
        normalisation_method="skip",
        imputation_method="knn_tn",
        knn_tn_k=4,
        knn_tn_distance="correlation",
    ),
)
assert p["knn_tn_k"] == 4
assert p["knn_tn_distance"] == "correlation"

p = show(
    "imputation: mindet (defaults)",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="mindet",
        normalisation_method="skip",
        imputation_method="mindet",
    ),
)
# NOTE: this uses the converter default q=0.01. The user flagged that
# md-form hides this parameter and prefers a different fixed value.
# Verify against the converter source before relying on the default.
print("mindet q default sent by builder:", p.get("q"))

p = show(
    "imputation: set to constant",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="const",
        normalisation_method="skip",
        imputation_method="set to constant",
        constant_value=42,
    ),
)
assert p["constant_value"] == 42

## 4. Peptide × filtration (PTM probability + by missing values)

In [ ]:
p = show(
    "peptide + by ptm localization probability",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="peptide PTM",
        entity_type="peptide",
        normalisation_method="skip",
        imputation_method="skip",
        filtration_method="by ptm localization probability",
        threshold=0.75,
    ),
)
assert p["filtration_methods_peptide"] == "by ptm localization probability"
assert p["threshold"] == 0.75

p = show(
    "peptide + by missing values (count criteria)",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="peptide missing count",
        entity_type="peptide",
        normalisation_method="skip",
        imputation_method="skip",
        filtration_method="by missing values",
        filter_valid_values_criteria="count",
        filter_threshold_count=3,
        filter_valid_values_logic="full experiment",
        experiment_design=DESIGN,
    ),
)
assert p["filter_valid_values_criteria"] == "count"
assert p["filter_threshold_count"] == 3
assert "filter_threshold_proportion" not in p

## 5. Gene × {cpm, ComBat-Seq, by minimum abundance}

In [ ]:
p = show(
    "gene + cpm (prior_count=2)",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="gene cpm",
        entity_type="gene",
        normalisation_method="cpm",
        imputation_method="skip",
        prior_count=2,
    ),
)
assert p["normalisation_methods_gene"] == "cpm"
assert p["prior_count"] == 2

p = show(
    "gene + ComBat-Seq batch correction",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="gene combat seq",
        entity_type="gene",
        normalisation_method="batch correction",
        imputation_method="skip",
        batch_correction_technique="combat seq",
        batch_variable_combat="batch",
        design_variables=[{"column": "condition", "type": "categorical"}],
        experiment_design=DESIGN,
    ),
)
assert p["batch_correction_technique_gene"] == "combat seq"
assert "mean_only" not in p
assert "reference_batch_combat" not in p

p = show(
    "gene + by minimum abundance (after CPM)",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="gene minabund",
        entity_type="gene",
        normalisation_method="cpm",
        imputation_method="skip",
        filtration_method="by minimum abundance",
        minimum_abundance_threshold=1.0,
        filter_valid_values_criteria="percentage",
        filter_threshold_proportion=0.5,
        filter_valid_values_logic="at least one condition",
        filter_based_on_condition="condition",
        experiment_design=DESIGN,
    ),
)
assert p["filtration_methods_gene"] == "by minimum abundance"
assert p["minimum_abundance_threshold"] == 1.0

## 6. `filter_only` classmethod (filtration-only NI; output stays INTENSITY)

In [ ]:
p = show(
    "protein filter_only (by missing values)",
    NormalisationImputationDataset.filter_only(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="protein filter only",
        entity_type="protein",
        filtration_method="by missing values",
        filter_valid_values_criteria="percentage",
        filter_threshold_proportion=0.5,
        filter_valid_values_logic="at least one condition",
        filter_based_on_condition="condition",
        experiment_design=DESIGN,
    ),
)
assert p["normalisation_methods_proteomics"] == "skip"
assert p["imputation_methods"] == "skip"
assert p["filtration_methods_protein"] == "by missing values"

## 7. Backward compat — legacy underscored aliases on input

In [ ]:
# Pass legacy underscored values; the wire payload should use canonical (spaced) form.
p = show(
    "legacy aliases -> canonical wire form",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="legacy",
        entity_type="peptide",
        normalisation_method="batch_correction",
        imputation_method="skip",
        filtration_method="ptm_localization_probability",
        batch_correction_technique="limma_remove_batch_effect",
        batch_variables=[{"column": "batch", "type": "categorical"}],
        experiment_design=DESIGN,
        threshold=0.5,
    ),
)
assert p["normalisation_methods_proteomics"] == "batch correction"
assert p["filtration_methods_peptide"] == "by ptm localization probability"
assert p["batch_correction_technique_proteomics"] == "limma remove batch effect"

## 8. Validation rejections

In [ ]:
def expect_error(label, ni, needle):
    try:
        ni.validate()
    except ValueError as e:
        msg = str(e)
        ok = needle in msg
        print(f"[{'OK' if ok else 'MISS'}] {label}: {msg}")
        assert ok, f"expected '{needle}' in error message"
        return
    raise AssertionError(f"{label} did not raise")


expect_error(
    "batch correction without technique",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="x",
        normalisation_method="batch correction",
        imputation_method="skip",
        experiment_design=DESIGN,
    ),
    "batch_correction_technique",
)

expect_error(
    "combat without batch_variable_combat",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="x",
        normalisation_method="batch correction",
        imputation_method="skip",
        batch_correction_technique="combat",
        experiment_design=DESIGN,
    ),
    "batch_variable_combat",
)

expect_error(
    "limma without batch_variables",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="x",
        normalisation_method="batch correction",
        imputation_method="skip",
        batch_correction_technique="limma remove batch effect",
        experiment_design=DESIGN,
    ),
    "batch_variables",
)

expect_error(
    "by missing values without criteria",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="x",
        entity_type="protein",
        normalisation_method="skip",
        imputation_method="skip",
        filtration_method="by missing values",
        experiment_design=DESIGN,
    ),
    "filter_valid_values_criteria",
)

expect_error(
    "by missing values rejected for gene",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="x",
        entity_type="gene",
        normalisation_method="skip",
        imputation_method="skip",
        filtration_method="by missing values",
    ),
    "filtration_method",
)

expect_error(
    "by minimum abundance rejected for protein",
    NormalisationImputationDataset(
        input_dataset_ids=DUMMY_INPUT,
        dataset_name="x",
        entity_type="protein",
        normalisation_method="skip",
        imputation_method="skip",
        filtration_method="by minimum abundance",
    ),
    "filtration_method",
)

## 9. (Optional) Live API smoke test

Uncomment and adjust to hit a real API. Requires `MD_API_BASE_URL` and `MD_AUTH_TOKEN` in env (or a `.env` file). Pick a known upload that already has a completed INTENSITY dataset.

`find_initial_dataset` now disambiguates: among INTENSITY datasets, it returns the unique one with empty `input_dataset_ids` (the upload-created original). NI/filter outputs always carry an upstream input, so they're skipped automatically. If that disambiguation still fails, fall back to `list_by_upload` and pick by name / state.

In [ ]:
# V2: list every dataset attached to an upload — useful when find_initial_dataset
# can't disambiguate, or when you want to pick a specific dataset id by hand.
from dotenv import load_dotenv

from md_python import MDClientV2

load_dotenv()
client = MDClientV2()

UPLOAD_ID = "a2f2dd97-c955-4845-a98c-628451140d1c"  # fill in

datasets = client.datasets.list_by_upload(UPLOAD_ID)
# Sort oldest -> newest so the upload-created dataset (no start time / earliest) is at the top.
datasets.sort(key=lambda d: d.job_run_start_time or "")

print(f"{len(datasets)} datasets for upload {UPLOAD_ID}\n")
print(
    f"{'id':36}  {'type':28}  {'state':10}  "
    f"{'started':19}  {'inputs':6}  name"
)
print("-" * 130)
for d in datasets:
    n_inputs = len(d.input_dataset_ids or [])
    started = (
        d.job_run_start_time.strftime("%Y-%m-%d %H:%M:%S")
        if d.job_run_start_time
        else "-"
    )
    print(
        f"{str(d.id):36}  {(d.type or ''):28}  {(d.state or ''):10}  "
        f"{started:19}  {n_inputs:>6d}  {d.name}"
    )

In [ ]:
ds = [x for x in datasets if x.id == UUID("faa1d362-903a-4f3e-b929-75c6ae1c7dbb")][0]
ds

In [ ]:
from dotenv import load_dotenv

from md_python import MDClientV2

load_dotenv()
client = MDClientV2()

UPLOAD_ID = "a2f2dd97-c955-4845-a98c-628451140d1c"  # fill in

# Disambiguates among multiple INTENSITY datasets by picking the one with no upstream inputs.
initial = ds #client.datasets.find_initial_dataset(UPLOAD_ID)
print("initial intensity dataset:", initial.id, initial.name)

# Pull the upload's sample metadata once, so the filter-only cell below can reuse it.
upload_sample_metadata = client.uploads.get_sample_metadata(UPLOAD_ID)
print("sample metadata columns:", list(upload_sample_metadata.to_columns().keys()))

ni = NormalisationImputationDataset(
    input_dataset_ids=[str(initial.id)],
    dataset_name="smoke v3 median + knn_tn",
    entity_type="protein",
    normalisation_method="median",
    imputation_method="knn_tn",
)
new_id = ni.run(client)
print("submitted:", new_id)
state = client.datasets.wait_until_complete(UPLOAD_ID, new_id)
print("final state:", state)

In [ ]:
ni

In [ ]:
# Filter-only example against a live API.
# Reuses `initial` and `upload_sample_metadata` from the cell above.
# experiment_design must be a {column: [values]} dict — that's exactly what
# SampleMetadata.to_columns() returns.

ni_filt = NormalisationImputationDataset.filter_only(
    input_dataset_ids=[str(initial.id)],
    dataset_name="smoke v3 protein filter_only",
    entity_type="protein",
    filtration_method="by missing values",
    filter_valid_values_criteria="percentage",
    filter_threshold_proportion=0.5,
    filter_valid_values_logic="at least one condition",
    filter_based_on_condition="condition",
    experiment_design=upload_sample_metadata.to_columns(),
)
filt_id = ni_filt.run(client)
print("submitted filter-only:", filt_id)
state = client.datasets.wait_until_complete(UPLOAD_ID, filt_id)
print("filter-only state:", state)